In [1]:
import os
os.chdir("/kaggle/working")
!pip uninstall -y torchao
!pip install -q deepspeed peft accelerate sentencepiece opensmile
if not os.path.exists("SpeechCueLLM"):
    !git clone -q https://github.com/zehuiwu/SpeechCueLLM.git

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 18.6 MB/s eta 0:00:00


In [2]:
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")
print(os.getcwd())

/kaggle/working/SpeechCueLLM/LLM_code


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
token = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download("meta-llama/Llama-3.2-3B-Instruct", local_dir="/kaggle/temp/llama32-3b",
                  token=token, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/kaggle/temp/llama32-3b'

In [4]:
import json, re
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

def sub(path, a, b):
    s = open(path).read()
    if a in s and b not in s:
        open(path, "w").write(s.replace(a, b))

M = "main.py"
sub(M, "model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
       "model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, torch_dtype=torch.float16, low_cpu_mem_usage=True)")
sub(M, '        deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
       '        deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
sub(M, "    model.gradient_checkpointing_enable()",
       "    model.gradient_checkpointing_enable()\n    model.enable_input_require_grads()")
sub(M, "    optimizer_grouped_parameters = getOptimizerGroup(model=model)\n",
       "    optimizer_grouped_parameters = getOptimizerGroup(model=model)\n"
       "    optimizer_grouped_parameters = [g for g in optimizer_grouped_parameters if len(g['params']) > 0]\n"
       "    print('optimizer groups:', [len(g['params']) for g in optimizer_grouped_parameters])\n")
sub(M, "    model_engine, optimizer, train_dataloader, lr_scheduler = deepspeed.initialize(",
       '    for _k in ("mics_shard_size", "mics_hierarchical_params_gather"):\n'
       '        deepspeed_config["zero_optimization"].pop(_k, None)\n'
       "    model_engine, optimizer, train_dataloader, lr_scheduler = deepspeed.initialize(")

p = "data_utils/deepspeed_config.json"
c = json.load(open(p))
c["bfloat16"]["enabled"] = False
c["fp16"]["enabled"] = True
c["zero_optimization"]["stage"] = 2
json.dump(c, open(p, "w"), indent=2)

EPOCHS, DATA_PCT, SEED = 3, 1.0, 11
sh = "train_and_inference.sh"
s = open(sh).read()
def setvar(name, val):
    global s
    s = re.sub(rf"^{name}=.*$", f"{name}={val}", s, count=1, flags=re.M)
setvar("MODEL_NAME", "'LLaMA3-instruct'"); setvar("Experiments_setting", "'lora'")
setvar("dataset", "'iemocap'")
setvar("num_train_epochs", EPOCHS); setvar("data_percent", DATA_PCT)
setvar("SEED", SEED); setvar("graphics_card", 2)
setvar("audio_description", "'True'"); setvar("audio_impression", "'False'")
setvar("audio_context", "'True'");     setvar("audio_only", "'False'")
setvar("task", "'B1_des_context'")
s = s.replace("MODEL_PATH='LLaMA3-instruct MODELPATH'", "MODEL_PATH='/kaggle/temp/llama32-3b'")
if "--gradient_checkpointing" not in s:
    s = s.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open(sh, "w").write(s)

!python -m py_compile main.py && echo "syntax ok"
!grep -n "MODEL_PATH='/kaggle" train_and_inference.sh
!grep -n "^dataset=\|^audio_\|^task=\|^num_train\|^data_percent\|MODEL_PATH='/kaggle" train_and_inference.sh

syntax ok
143:        MODEL_PATH='/kaggle/temp/llama32-3b'
21:dataset='iemocap'
25:audio_description='True'
26:audio_impression='False'
27:audio_context='True'
28:audio_only='False'
32:num_train_epochs=3
47:task='B1_des_context'
55:data_percent=1.0
143:        MODEL_PATH='/kaggle/temp/llama32-3b'


In [5]:
!python data_process.py --dataset iemocap --historical_window 12 --audio_description True --audio_impression False --audio_only False --audio_context True --experiments_setting lora
!find ../PROCESSED_DATASET -name "*.json" | head

../PROCESSED_DATASET/iemocap/window/True_False
../PROCESSED_DATASET/iemocap/window/True_False/valid.json
../PROCESSED_DATASET/iemocap/window/True_False/train.json
../PROCESSED_DATASET/iemocap/window/True_False/test.json


In [6]:
import json
with open("../PROCESSED_DATASET/iemocap/window/True_False/train.json") as f:
    ex = json.loads(f.readline())
print(ex.keys())
print(str(ex)[:3000])

dict_keys(['path', 'input', 'target'])
{'path': '../data/IEMOCAP_full_release/Session3/sentences/wav/Ses03F_impro06/Ses03F_impro06_M000.wav', 'input': 'Now you are expert of sentiment and emotional analysis.The following conversation noted between \'### ###\' involves several speakers. The last three utterances are followed by its speech features. ### \t Speaker_0:"How you doing?" (medium pitch with very_high variation) ### Target speech characteristics: very low volume with very low variation, moderate pitch with very high variation, low speaking rate. Please select the emotional label of < Speaker_0:"How you doing?"> from <happy, sad, neutral, angry, excited, frustrated> based on both the context and audio features. Respond with just one label:', 'target': 'neutral'}


In [7]:
import subprocess, time, re, shutil
EPOCHS = 3
proc = subprocess.Popen("timeout 11h bash train_and_inference.sh > ../run_log.txt 2>&1", shell=True)
while proc.poll() is None:
    time.sleep(60)
    txt = open("../run_log.txt", errors="ignore").read().replace("\r", "\n")
    done = [h for h in re.findall(r"history: \[(.*?)\]", txt) if len(h.split(",")) >= EPOCHS]
    if len(done) >= 2:
        time.sleep(20)
        subprocess.run("pkill -f main.py; pkill -f deepspeed", shell=True)
        break
txt = open("../run_log.txt", errors="ignore").read().replace("\r", "\n")
res = "\n".join(l for l in txt.split("\n") if re.search(r"best_f1|current_f1|history", l))
open("/kaggle/working/results_B1.txt", "w").write(res)
shutil.copy("../run_log.txt", "/kaggle/working/run_log_B1.txt")
print(res, flush=True)

best_f1: 60.96
current_f1: 60.96
history: [60.96]
best_f1: 61.61
current_f1: 61.61
history: [61.61]
best_f1: 67.005
current_f1: 67.005
history: [60.96, 67.005]
Runing epoch2 / 3:   0%|          | 0/2582 [00:00<?, ?it/s]best_f1: 67.244
current_f1: 67.244
history: [61.61, 67.244]
100%|█████████▉| 202/203 [06:25<00:02,  2.00s/it]best_f1: 68.052
current_f1: 68.052
history: [60.96, 67.005, 68.052]
best_f1: 67.244
current_f1: 67.055
history: [61.61, 67.244, 67.055]
